# R7.2 | ADT独立流轴验证：全量共域评估 + 2023失败轨迹断面保全审计

**单Notebook / Python / WSL / VS Code / 不重训 / 不重跑1993–2022 ADT追踪。**

使用既有 R7.0 追踪数据和结果。第一次校核全量 10,279 日的共同空间范围误差（而非先前 744 日抽样）；第二次对 **2023 年追踪失败日期**重新运行原始同参数、同 V6 Python 算法，但仅将失败前的**独立 ADT 部分路径**作为候选进行断面覆盖审计。**绝不将不完整流轴标为完整成功；绝不以 GLORYS 或预测误差筛选 ADT 路径。**

警告：ADT 为0.125°卫星表层锋面代理，GLORYS为50m流速核；二者定义不同，GLORYS可能同化卫星海面高度。2023部分路径分析为探索性敏感性，必须与主分析分开。请不要将两者混同。

In [ ]:
# 0. 路径及配置（建议首次只运行到数据路径检查）
from pathlib import Path
from dataclasses import dataclass
import os, re, io, json, zipfile, hashlib, traceback, time, warnings, math
import numpy as np, pandas as pd
from scipy.spatial import cKDTree
from IPython.display import display, FileLink
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT=Path(os.environ.get('KUROSHIO_ROOT','/mnt/d/KuroshioDaily1993_2025'))
R70_ROOT=ROOT/'R70_ADT_Independent_Python'
ADT_AX=R70_ROOT/'adt_axes_npz'
GLO_AX=ROOT/'final/tracks_final'
R70_RESULTS=ROOT/'R70_PY_上传给ChatGPT_ADT独立参考验证完整结果.zip'
R70_QC=ROOT/'R70_PY_上传给ChatGPT_日期覆盖与流轴追踪QC.zip'
OUT=ROOT/'R72_ADT_CompleteDomain_2023PartialAudit'
TBL=OUT/'tables';FIG=OUT/'figures_en';PART=OUT/'adt_partial_2023';REP=OUT/'reports'
for p in (OUT,TBL,FIG,PART,REP):p.mkdir(parents=True,exist_ok=True)

# 不变更原模型、ADT字段或V6参数；以下两个开关控制耗时
DO_ALL_COMMON=True   # 10,279条有效日期全量共域评分；不会重新追踪ADT
DO_2023_PARTIAL=True # 仅重新生成被原协议拒绝的2023部分轨迹（原算法/原参数）
PARTIAL_MAX_DAYS=None # 首次试运行可设10，之后必须恢复None才能宣称全年审计
DOMAIN_END_LON=144.3
DENSIFY_KM=5.0
BOOTSTRAP_REPS=1200
BOOTSTRAP_SEED=20261016
MIN_SECTION_REMAINING_POINTS=3 # 断面交点后至少剩余3点，不把终止处不稳定交点算合格

for desc,p in [('R70_RESULTS',R70_RESULTS),('R70_QC',R70_QC),('ADT NPZ',ADT_AX),('GLORYS NPZ',GLO_AX)]:
 print(desc, 'found=',p.exists(),p)
if not R70_RESULTS.exists() or not R70_QC.exists():
 raise FileNotFoundError('缺少R7.0两份结果ZIP。不要执行后续单元。')
if DO_ALL_COMMON and (not ADT_AX.is_dir() or not GLO_AX.is_dir()):
 raise FileNotFoundError('全量共域评分需要本地ADT_AX及GLO_AX NPZ原轨迹。')


In [ ]:
# 1. R7.0真实归档与数据完整性

def sha256(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
 return h.hexdigest()

def zip_csv(z,name):
 with z.open(name) as f:return pd.read_csv(f)
with zipfile.ZipFile(R70_RESULTS) as z:
 assert z.testzip() is None
 daily=zip_csv(z,'daily/每日ADT_vs_GLORYS完整流轴空间误差.csv')
 section=zip_csv(z,'daily/六断面逐日ADT与GLORYS成对误差.csv')
with zipfile.ZipFile(R70_QC) as z:
 assert z.testzip() is None
 qc=zip_csv(z,'05_ADT逐日追踪及QC报告.csv')

print('R70 main sha256',sha256(R70_RESULTS),'QC sha256',sha256(R70_QC))
print('QC date count:',len(qc),'success',sum(qc.status=='success'),'failed',sum(qc.status!='success'))
print('Daily paired',len(daily),'Section daily',len(section))
assert len(qc)==11322 and (qc.status=='success').sum()==10279, '当前上传R70数据版本与预期不同，请复核后调整，不可悄然继续'
assert len(daily)==10279, '缺少权威10,279条逐日记录'
print('Daily columns:',daily.columns.tolist());print('Section columns:',section.columns.tolist())

def save_csv(df,name):
 p=TBL/name; df.to_csv(p,index=False,encoding='utf-8-sig');return p
save_csv(qc.groupby([qc.date.str[:4],'status']).size().rename('n').reset_index(),'01_ADT年度QC状态.csv')


In [ ]:
# 2. 共域流轴评分器：沿用R7.1的球面最近距离定义，不接触真值或预测生成
EARTH_RADIUS_KM=6371.0088

def xyz(x):
 a=np.deg2rad(np.asarray(x,float));lo=a[:,0];la=a[:,1]
 return EARTH_RADIUS_KM*np.column_stack([np.cos(la)*np.cos(lo),np.cos(la)*np.sin(lo),np.sin(la)])
def gc_km(chord):return 2*EARTH_RADIUS_KM*np.arcsin(np.clip(np.asarray(chord)/(2*EARTH_RADIUS_KM),0,1))
def densify(x,spacing=DENSIFY_KM):
 x=np.asarray(x,float)
 if len(x)<2 or not np.isfinite(x).all():raise ValueError('Invalid axis')
 d=gc_km(np.linalg.norm(np.diff(xyz(x),axis=0),axis=1));
 if (d>250).any():raise ValueError('track contains >250 km discontinuity')
 ret=[x[0]]
 for a,b,dist in zip(x[:-1],x[1:],d):
  n=max(1,int(np.ceil(dist/spacing)))
  ret.extend(a+(b-a)*(j/n) for j in range(1,n+1))
 return np.asarray(ret)
def score(a,b):
 aa=densify(a);bb=densify(b);u=xyz(aa);v=xyz(bb)
 dist=np.r_[gc_km(cKDTree(v).query(u)[0]),gc_km(cKDTree(u).query(v)[0])]
 return {'rmse_km':float(np.sqrt(np.mean(dist**2))), 'mae_km':float(np.mean(dist)), 'p90_km':float(np.quantile(dist,.9))}
def east_crossings(x,limit=DOMAIN_END_LON):
 return int(np.sum((x[:-1,0]<limit)&(x[1:,0]>=limit)))

def clip_lon(x,limit=DOMAIN_END_LON):
 for i in range(1,len(x)):
  lo,hi=x[i-1,0],x[i,0]
  if lo<limit<=hi:
   frac=(limit-lo)/(hi-lo)
   return np.vstack((x[:i],x[i-1]+frac*(x[i]-x[i-1])))
 raise ValueError('does not reach prescribed east longitude')
def load_axis(p,kind):
 with np.load(p,allow_pickle=False) as z:
  lon=z['axis_lon'] if kind=='ADT' else z['lon']
  lat=z['axis_lat'] if kind=='ADT' else z['lat']
 x=np.column_stack((lon,lat)).astype(float)
 if not np.isfinite(x).all():raise ValueError('axis contains missing coordinates')
 return x

def cluster_year_ci(df,column='common_rmse_km',B=BOOTSTRAP_REPS,seed=BOOTSTRAP_SEED):
 # 以年份为独立重采样簇，不把连续每日数据错误当作独立样本。
 annual=df.groupby('year')[column].mean().dropna().to_numpy(float)
 if len(annual)<4:return [float('nan'),float('nan')]
 r=np.random.default_rng(seed)
 idx=r.integers(0,len(annual),size=(B,len(annual)))
 x=annual[idx].mean(axis=1)
 return list(np.quantile(x,[.025,.975]).astype(float))


In [ ]:
# 3. 全部10,279天共域评估+年度分层bootstrap（随时可重跑，逐日CSV断点恢复）
CACHE=TBL/'02_完整共域逐日误差.csv'
accepted=qc.loc[qc.status=='success',['date']].drop_duplicates().sort_values('date')
assert len(accepted)==10279
if CACHE.exists():
 known=pd.read_csv(CACHE)
else:
 known=pd.DataFrame()
old_dates=set(known.date.astype(str)) if not known.empty else set()
rows=[]
if DO_ALL_COMMON:
 for i,day in enumerate(accepted.date.astype(str),1):
  if day in old_dates:continue
  stem=day.replace('-','')
  try:
   a=load_axis(ADT_AX/(stem+'.npz'),'ADT')
   b=load_axis(GLO_AX/(stem+'.npz'),'GLORYS')
   original=score(a,b);common=score(clip_lon(a),clip_lon(b))
   rows.append({'date':day,'year':int(day[:4]), 'status':'ok',
     'original_rmse_km':original['rmse_km'],'common_rmse_km':common['rmse_km'],
     'original_mae_km':original['mae_km'],'common_mae_km':common['mae_km'],
     'common_p90_km':common['p90_km'], 'delta_rmse_km':common['rmse_km']-original['rmse_km'],
     'adt_east_crossings':east_crossings(a),'glorys_east_crossings':east_crossings(b)})
  except Exception as e:rows.append({'date':day,'year':int(day[:4]),'status':'failed','reason':str(e)})
  if len(rows)>=60 or i==len(accepted):
   piece=pd.DataFrame(rows);known=pd.concat([known,piece],ignore_index=True)
   known=known.drop_duplicates('date',keep='last').sort_values('date')
   save_csv(known,'02_完整共域逐日误差.csv');rows.clear()
  if i%400==0:print('[Full common domain]',i,'/',len(accepted),'processed, existing',len(old_dates),flush=True)
else:
 print('DO_ALL_COMMON=False: reading previous saved results only')
if CACHE.exists():
 full=pd.read_csv(CACHE);v=full[full.status=='ok'].copy()
 print('Complete shared range:',len(v),'/',len(accepted),'valid')
 if len(v):
  metrics=v[['original_rmse_km','common_rmse_km','original_mae_km','common_mae_km','delta_rmse_km']].agg(['mean','median']).T
  display(metrics.round(3))
  yy=v.groupby('year').agg(n=('date','size'),original_rmse_km=('original_rmse_km','mean'),common_rmse_km=('common_rmse_km','mean'),delta_km=('delta_rmse_km','mean')).reset_index()
  save_csv(yy,'03_共域误差年度汇总.csv')
  audit={'days_ok':int(len(v)),'days_total':int(len(accepted)),
    'full_range_rmse_mean':float(v.original_rmse_km.mean()),
    'common_rmse_mean':float(v.common_rmse_km.mean()),
    'common_mae_mean':float(v.common_mae_km.mean()),
    'common_rmse_yearbalanced_mean':float(yy.common_rmse_km.mean()),
    'common_rmse_yearcluster_ci95':cluster_year_ci(v),
    'delta_rmse_yearcluster_ci95':cluster_year_ci(v,'delta_rmse_km'),
    'common_definition':'first crossing of 144.3E, symmetric great-circle-nearest polyline RMSE',
    'no_forecast_data_used':True}
  if {'adt_east_crossings','glorys_east_crossings'}.issubset(v.columns):
   sub=v[(v.adt_east_crossings==1)&(v.glorys_east_crossings==1)]
   audit['unique_first_crossing_days']=int(len(sub))
   audit['unique_first_crossing_mean_rmse_km']=float(sub.common_rmse_km.mean()) if len(sub) else None
   audit['multi_crossing_days']=int(len(v)-len(sub))
  (REP/'R72_完整共域审计.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
  print('Year clustered bootstrap 95% CI',audit['common_rmse_yearcluster_ci95'])


In [ ]:
# 4. 2023 ADT失败：位置和原因（不需要重新跑NetCDF）
failure=qc[(qc.date.astype(str).str.startswith('2023')) & (qc.status!='success')].copy()
assert len(failure)==274
failure['month']=failure.date.astype(str).str[:7]
failure['failure_region']=pd.cut(failure.last_lon,[-np.inf,124,128.5,132,137,144.3,np.inf],
 labels=['Taiwan','ECS','Tokara','SouthJapan','Downstream','Beyond144.3E'])
monthly2023=qc[qc.date.astype(str).str.startswith('2023')].groupby([qc.date.str[:7],'status']).size().unstack(fill_value=0)
monthly2023['total']=monthly2023.sum(axis=1)
monthly2023['success_rate']=monthly2023.get('success',0)/monthly2023.total
save_csv(failure,'04_2023追踪失败日定位及原因.csv')
save_csv(monthly2023.reset_index().rename(columns={'date':'month'}),'05_2023逐月有效率.csv')
print('2023 failed',len(failure),'downstream 137-144.3E:',int(((failure.last_lon>=137)&(failure.last_lon<144.3)).sum()))
print(failure.groupby(['month','failure_region'],observed=False).size().unstack(fill_value=0).to_string())


## 2023年 ADT-only 部分路径实验（探索性，禁止并入主验证）

以下完全复制上一版 R7.0 Python V6 追踪核心，**没有调松任何 V6 物理参数，也不使用 GLORYS 位置协助跟踪**。原始拒绝日的中间轨迹本来未保存，故仅对2023年274个失败日重跑并缓存失败前的部分轨迹。所有输出必须标记 `partial_exploratory`。

In [ ]:
"""Independent satellite ADT Kuroshio axis V6 Python port.
Port of user-supplied kuroshio_axis_ADT_final_v4_seedfix_v6.m, 2026-10.
Topological penalty, seed DP, ridge-band, local multistep recovery, and ridge-constrained corners retained.
Numerical equivalence is NOT assumed; validate with MATLAB on same snapshots before quantitative publication.
"""
from __future__ import annotations
import math, os, re, warnings, traceback
from dataclasses import dataclass
from pathlib import Path
from functools import lru_cache
import numpy as np
import pandas as pd
import xarray as xr
from scipy.ndimage import gaussian_filter, convolve
from scipy.interpolate import RegularGridInterpolator
import contourpy

KMLAT=111.32

def geo(x,y,x2,y2):
    return np.hypot((np.asarray(x2)-x)*KMLAT*np.cos(np.deg2rad((np.asarray(y2)+y)/2)),(np.asarray(y2)-y)*KMLAT)
def dist_points(x,y,xp,yp): return geo(x,y,xp,yp)
def km2deg(dx,dy,lat):
    return np.asarray(dx)/max(1e-9,KMLAT*math.cos(math.radians(float(lat)))),np.asarray(dy)/KMLAT
def tangent(x1,y1,x2,y2,lat): return np.array([(x2-x1)*KMLAT*np.cos(np.deg2rad(lat)),(y2-y1)*KMLAT],float)
def unit(a):
    a=np.asarray(a,float);n=np.linalg.norm(a)
    return a/n if n>1e-10 and np.isfinite(n) else np.array([np.nan,np.nan])
def cosangle(a,b): return float(np.clip(np.dot(a,b),-1,1))
def ang(a,b):return math.degrees(math.acos(cosangle(a,b)))
def turn3(x1,y1,x2,y2,x3,y3):
    v1=unit(tangent(x1,y1,x2,y2,(y1+y2)/2));v2=unit(tangent(x2,y2,x3,y3,(y2+y3)/2))
    return ang(v1,v2) if np.isfinite(v1).all() and np.isfinite(v2).all() else np.nan
def turning(x,y):
    z=np.full(len(x),np.nan)
    for i in range(1,len(x)-1):z[i]=turn3(x[i-1],y[i-1],x[i],y[i],x[i+1],y[i+1])
    return z

def ridge_band(g,valid,sampling=3.,band=.8,global_ratio=.55,neighbor=15.):
    g=np.asarray(g,float);valid=np.asarray(valid,bool);n=len(g)
    if not valid.any():return np.array([],int)
    peaks=[]
    if n>=2:
        if valid[0] and (not valid[1] or g[0]>=g[1]):peaks.append(0)
        if valid[-1] and (not valid[-2] or g[-1]>=g[-2]):peaks.append(n-1)
    for j in range(1,n-1):
        if valid[j-1] and valid[j] and valid[j+1] and g[j]>=g[j-1] and g[j]>=g[j+1]:peaks.append(j)
    gmax=np.max(g[valid]);
    if not np.isfinite(gmax) or gmax<=0:return np.array([],int)
    mask=valid&(g>=global_ratio*gmax);rad=max(1,int(round(neighbor/max(sampling,1e-8))))
    for j in peaks:
        idx=np.arange(max(0,j-rad),min(n,j+rad+1));mask[idx]|=valid[idx]&(g[idx]>=band*g[j])
    cand=np.flatnonzero(mask)
    return cand if len(cand) else np.array([np.flatnonzero(valid)[np.argmax(g[valid])]])

class Fields:
    def __init__(self,lon,lat,a,sigma_fine=1.,sigma_coarse=3.,build_topo=True):
        self.lon=np.asarray(lon,float);self.lat=np.asarray(lat,float)
        self.A=np.asarray(a,float)
        assert self.A.shape==(len(lat),len(lon)),(self.A.shape,len(lat),len(lon))
        assert np.all(np.diff(lon)>0) and np.all(np.diff(lat)>0)
        valid=np.isfinite(self.A)
        def filt(sig):
            num=gaussian_filter(np.where(valid,self.A,0),sig,mode='nearest')
            den=gaussian_filter(valid.astype(float),sig,mode='nearest')
            return np.where(den>=.6,num/np.maximum(den,1e-12),np.nan)
        self.fine=filt(sigma_fine);self.coarse=filt(sigma_coarse)
        # MATLAB: interiorOcean=conv2(double(isfinite(ADT_fine)),ones(3,3),'same')>=8
        self.interior=convolve(np.isfinite(self.fine).astype(np.int16),np.ones((3,3),dtype=np.int16),mode='constant',cval=0)>=8
        self.gx=self.gy=self.g=self.gc=None
        # MATLAB gradient dA/dlon, dA/dlat, converted m/km.
        dy,dx=np.gradient(self.fine,self.lat,self.lon)
        dyc,dxc=np.gradient(self.coarse,self.lat,self.lon)
        gridcos=np.maximum(.1,np.cos(np.deg2rad(self.lat[:,None])))
        self.gx=dx/(KMLAT*gridcos);self.gy=dy/KMLAT
        self.g=np.hypot(self.gx,self.gy)
        self.gc=np.hypot(dxc/(KMLAT*gridcos),dyc/KMLAT)
        self.g95=max(float(np.nanpercentile(self.g,95)),1e-10)
        self.gc95=max(float(np.nanpercentile(self.gc,95)),1e-10)
        self.i={}
        for key,v in [('a',self.fine),('g',self.g),('gc',self.gc),('gx',self.gx),('gy',self.gy)]:
            self.i[key]=RegularGridInterpolator((self.lat,self.lon),v,bounds_error=False,fill_value=np.nan)
        self.topo=Topology(self.lon,self.lat,self.fine) if build_topo else None
    def get(self,name,ys,xs):
        yy,xx=np.broadcast_arrays(np.asarray(ys,float),np.asarray(xs,float))
        z=self.i[name](np.column_stack([yy.ravel(),xx.ravel()])).reshape(yy.shape)
        return float(z) if z.ndim==0 else z
    def inside(self,x,y):return self.lon[0]<=x<=self.lon[-1] and self.lat[0]<=y<=self.lat[-1]
    def profile(self,x0,y0,normal,width,sampling,attrs=('g','gc','a','gx','gy')):
        off=np.arange(-width,width+1e-7,sampling)
        dx,dy=km2deg(off*normal[0],off*normal[1],y0)
        xx=x0+dx;yy=y0+dy
        results={k:self.get(k,yy,xx) for k in attrs}
        return off,xx,yy,results

class Topology:
    """MATLAB contourc-like level library at 2-cm increments, small shapes only."""
    def __init__(self,lon,lat,z,step=.02,minlen=80.):
        finite=z[np.isfinite(z)]
        self.step=step;self.segments={}
        if len(finite)<10:return
        lev1=np.ceil(finite.min()/step)*step;lev2=np.floor(finite.max()/step)*step
        if lev2-lev1<step:return
        cg=contourpy.contour_generator(x=lon,y=lat,z=np.ma.masked_invalid(z),name='serial',corner_mask=False)
        levels=np.arange(lev1,lev2+step*.2,step)
        for li,v in enumerate(levels):
            seg=[]
            for xy in cg.lines(float(v)):
                if len(xy)<3:continue
                x,y=xy[:,0],xy[:,1]
                length=np.sum(geo(x[:-1],y[:-1],x[1:],y[1:]))
                if length<minlen:continue
                seg.append((xy[::3].copy(),bool(geo(x[0],y[0],x[-1],y[-1])<25)))
            if seg:self.segments[li]=seg
        self.start=float(lev1);self.n=len(levels)
    def query(self,x,y,a,match=18.):
        if not np.isfinite(a) or not hasattr(self,'n'):return False,False
        ind=int(np.clip(np.round((a-self.start)/self.step),0,self.n-1))
        if ind not in self.segments:return False,False
        best=float('inf');closed=False
        for pts,isclosed in self.segments[ind]:
            d=float(np.min(geo(x,y,pts[:,0],pts[:,1])))
            if d<best:best=d;closed=isclosed
        return bool(closed), bool(best<=match)

@dataclass
class V6:
    max_points:int=650
    stop_lon:float=144.3
    stop_lat:float=29.0
    use_topology:bool=True
    regularize:bool=True
    enable_recovery:bool=True
    # Original MATLAB V6 parameters retained in port; do not tune on ADT–GLORYS agreement.

GATES=np.array([[128,131.5,27,31.8],[133,138.5,27,34.8],[140,144.8,29,37.5]])
def gate_pass(x,y,g,margin):return x>=g[1]-margin and g[2]-.8<=y<=g[3]+.8
def guide(x,y,stage):
    if stage>=len(GATES):return np.array([1.,0.])
    a,b,c,d=GATES[stage];xt=np.clip(x,a,b);yt=np.clip(y,c,d)
    v=np.array([(xt-x)*KMLAT*np.cos(np.deg2rad(y)),(yt-y)*KMLAT])
    return unit(v) if np.linalg.norm(v)>1e-6 else np.array([1.,0.])

def taiwan_seed(F):
    latidx=np.where((F.lat>=22)&(F.lat<=24.4))[0]
    lonidx=np.where((F.lon>=121)&(F.lon<=124))[0]
    if len(latidx)<6 or len(lonidx)<3:raise RuntimeError('insufficient Taiwan seed grid')
    C=[];GG=F.g[np.ix_(latidx,lonidx)];GC=F.gc[np.ix_(latidx,lonidx)]
    gs=max(np.nanpercentile(GG,90),1e-10);gcs=max(np.nanpercentile(GC,90),1e-10)
    for iy in latidx:
        g=F.g[iy,lonidx]
        valid=F.interior[iy,lonidx]&np.isfinite(g)&np.isfinite(F.gc[iy,lonidx])&np.isfinite(F.fine[iy,lonidx])&np.isfinite(F.gx[iy,lonidx])&np.isfinite(F.gy[iy,lonidx])
        peaks=np.zeros(len(lonidx),bool)
        peaks[1:-1]=(g[1:-1]>=g[:-2])&(g[1:-1]>=g[2:])
        jj=np.flatnonzero(valid&peaks)
        if not len(jj):jj=np.flatnonzero(valid)
        rows=[]
        for j in jj:
            ix=lonidx[j];gxx=F.gx[iy,ix];gyy=F.gy[iy,ix];ng=np.hypot(gxx,gyy)
            if ng<=1e-10 or gxx/ng<.2:continue
            closed,known=F.topo.query(F.lon[ix],F.lat[iy],F.fine[iy,ix]) if F.topo else (False,False)
            penalty=2.5 if closed and known else (.15 if not known else 0)
            score=min(F.g[iy,ix]/gs,1.8)+.75*min(F.gc[iy,ix]/gcs,1.8)+.35*gxx/ng-penalty
            rows.append([F.lon[ix],F.lat[iy],F.g[iy,ix],F.gc[iy,ix],F.fine[iy,ix],gxx/ng,score])
        C.append(sorted(rows,key=lambda a:-a[6])[:6])
    S=[];P=[];L=[]
    for ri,cur in enumerate(C):
        sco=np.array([z[6] for z in cur],float);back=np.full(len(cur),-1,int);plen=np.ones(len(cur),int)
        if ri and len(C[ri-1]):
            for j,c in enumerate(cur):
                for k,p in enumerate(C[ri-1]):
                    dx=abs(c[0]-p[0])*KMLAT*np.cos(np.deg2rad((c[1]+p[1])/2))
                    if dx>42:continue
                    d=np.hypot(dx,KMLAT*abs(c[1]-p[1]))
                    trans=.85*(dx/42)**2+.30*(abs(c[4]-p[4])/.08)**2+.05*(d/20)**2
                    v=S[-1][k]+c[6]-trans
                    if v>sco[j]:sco[j]=v;back[j]=k;plen[j]=L[-1][k]+1
        S.append(sco);P.append(back);L.append(plen)
    best=(-np.inf,-1,-1)
    for i,Ci in enumerate(C):
        for j in range(len(Ci)):
            if L[i][j]>=6 and S[i][j]+.35*L[i][j]>best[0]:best=(S[i][j]+.35*L[i][j],i,j)
    if best[1]<0:raise RuntimeError('V6 Taiwan entrance DP no 6-point continuous northward ridge')
    i,j=best[1:];seg=[]
    for _ in range(int(L[i][j])):
        seg.append(C[i][j][:2]);j0=P[i][j];i-=1;j=j0
        if j<0:break
    seg=np.asarray(seg[::-1],float)
    if len(seg)<6:raise RuntimeError('short Taiwan entrance seed')
    return seg

def support_ahead(F,x,y,t,step=12.,width=28.,spacing=4.):
    dx,dy=km2deg(step*t[0],step*t[1],y);xp=x+dx;yp=y+dy
    if not F.inside(xp,yp):return 0.
    _,xx,yy,z=F.profile(xp,yp,[-t[1],t[0]],width,spacing,('g','gc'))
    valid=np.isfinite(z['g'])&np.isfinite(z['gc'])&(z['g']>0)
    if not valid.any():return 0.
    scores=.55*np.minimum(z['g'][valid]/F.g95,1.5)+.45*np.minimum(z['gc'][valid]/F.gc95,1.5)
    return float(np.max(scores))
def support_multi(F,x,y,t,nsteps=3,step=12.,width=30.,spacing=4.):
    vals=[]
    for _ in range(nsteps):
        dx,dy=km2deg(step*t[0],step*t[1],y);xp=x+dx;yp=y+dy
        if not F.inside(xp,yp):return 0.
        _,xx,yy,z=F.profile(xp,yp,[-t[1],t[0]],width,spacing,('g','gc'))
        good=np.isfinite(z['g'])&np.isfinite(z['gc'])&(z['g']>0)
        if not good.any():return 0.
        cand=np.flatnonzero(good)
        support=.55*np.minimum(z['g'][cand]/F.g95,1.5)+.45*np.minimum(z['gc'][cand]/F.gc95,1.5)
        j=cand[np.argmax(support)];vals.append(np.max(support));x,y=xx[j],yy[j]
    return float(min(vals))
def mini_track(F,x,y,t):
    xstart,ystart=x,y;vals=[]
    for _ in range(3):
        dx,dy=km2deg(14*t[0],14*t[1],y);xp=x+dx;yp=y+dy
        if not F.inside(xp,yp):return 0.,-np.inf
        off,xx,yy,z=F.profile(xp,yp,[-t[1],t[0]],34,4)
        ok=np.isfinite(z['g'])&np.isfinite(z['gc'])&np.isfinite(z['gx'])&np.isfinite(z['gy'])&(z['g']>0)
        cand=ridge_band(z['g'],ok,4,.76,.45,20)
        best=(-np.inf,None,None,None)
        for j in cand:
            tc=unit([-z['gy'][j],z['gx'][j]])
            if not np.isfinite(tc).all() or cosangle(t,tc)<np.cos(np.deg2rad(100)):continue
            sr=.55*min(z['g'][j]/F.g95,1.5)+.45*min(z['gc'][j]/F.gc95,1.5)
            score=sr+.2*cosangle(t,tc)-.12*(off[j]/34)**2
            if score>best[0]:best=(score,j,tc,sr)
        if best[1] is None:return 0.,-np.inf
        j=best[1];x,y=xx[j],yy[j];vals.append(best[3]);t=unit(.45*t+.55*best[2])
    return float(min(vals)),float((x-xstart)*KMLAT*np.cos(np.deg2rad((y+ystart)/2)))

def recover(F,x,y,a0,t,g,axis,stage):
    """V6 small forward-sector reacquisition + independent 3-step ridge persistence test."""
    if len(axis)<2:return None
    recent=unit(tangent(axis[max(0,len(axis)-6)][0],axis[max(0,len(axis)-6)][1],axis[-1][0],axis[-1][1],y))
    if not np.isfinite(recent).all():recent=t
    if cosangle(recent,t)<0:recent=-recent
    base=unit(.9*(.55*t+.45*recent)+.1*g)
    if not np.isfinite(base).all():base=t
    out=None;bestscore=-np.inf
    for fwd in (12,18,24,30,36,48):
        dx,dy=km2deg(fwd*base[0],fwd*base[1],y);xp=x+dx;yp=y+dy
        if not F.inside(xp,yp):continue
        off,xx,yy,z=F.profile(xp,yp,[-base[1],base[0]],84,4)
        good=np.isfinite(z['g'])&np.isfinite(z['gc'])&np.isfinite(z['a'])&np.isfinite(z['gx'])&np.isfinite(z['gy'])&(z['g']>0)
        cand=ridge_band(z['g'],good,4,.76,.45,20)
        for j in cand:
            x1,y1=xx[j],yy[j];gg=z['g'][j];gc=z['gc'][j];a=z['a'][j]
            tc=unit([-z['gy'][j],z['gx'][j]])
            if not np.isfinite(tc).all():continue
            jump=float(geo(x,y,x1,y1));mv=unit(tangent(x,y,x1,y1,(y+y1)/2))
            if jump<7.8 or jump>65 or not np.isfinite(mv).all():continue
            turn=ang(t,tc);move_turn=ang(base,mv)
            if turn>125 or move_turn>80:continue
            gd=cosangle(tc,g)
            if x<124 and (tc[1]<-.05 or gd<-.45):continue
            if 124<=x<130 and gd<-.60:continue
            if x>=130 and (tc[0]<-.60 or gd<-.65):continue
            if 128.5<=x<=132.5:
                if tc[0]<-.45:continue
                if len(axis)>8:
                    zold=axis[-8];dxnet=(x1-zold[0])*KMLAT*np.cos(np.deg2rad((y1+zold[1])/2))
                    if dxnet<-25:continue
            if stage==0 and 128.7<=x<=131.7:
                if y1>=30.45 and x1<130.25:continue
                if y>=29.7 and tc[0]<-.02:continue
                if y1>=29.7 and x1<130.05 and tc[0]<.18:continue
            if 122<=x<=126.2 and 24.4<=y<=26.9 and len(axis)>7:
                old=axis[-7];chord=geo(old[0],old[1],x1,y1)
                seg=np.asarray(axis[-7:]+[[x1,y1]])
                plen=float(np.sum(geo(seg[:-1,0],seg[:-1,1],seg[1:,0],seg[1:,1])))
                eff=chord/max(plen,1e-10);dxnet=(x1-old[0])*KMLAT*np.cos(np.deg2rad((y1+old[1])/2))
                if eff<.48 or dxnet<-12 or turn>68:continue
            gr=gg/F.g95;gcr=gc/F.gc95
            if gr<.18 or gcr<.18:continue
            dhard=min(.18,.12+.0015*max(0,jump-12));da=abs(a-a0)
            if da>dhard:continue
            lp=0.
            if len(axis)>18:
                old=np.asarray(axis[:-12]);dm=float(np.min(geo(x1,y1,old[:,0],old[:,1])))
                if dm<24:continue
                if dm<45:lp=(45-dm)/21
            closed,known=F.topo.query(x1,y1,a,18) if F.topo and x>=126 else (False,False)
            tp=1.1 if known and closed else (.1 if not known and F.topo and x>=126 else 0)
            supp=support_ahead(F,x1,y1,tc)
            if supp<.18:continue
            mini,dxmini=mini_track(F,x1,y1,tc)
            if mini<.20:continue
            if 128.5<=x<=132.5 and dxmini<-5:continue
            sr=.45*supp+.55*mini
            sc=min(gr,1.7)+.7*min(gcr,1.7)+.55*min(sr,1.5)+.1*max(gd,0)
            sc-=.22*(off[j]/84)**2+.55*(turn/90)**2+.35*(move_turn/70)**2
            sc-=.55*(da/.12)**2+.20*((jump-12)/55)**2+.30*(max(0,.1-gd)/1.1)**2+1.6*lp+tp
            if sc>bestscore:bestscore=sc;out=(float(x1),float(y1),float(gg),float(a),float(off[j]),float(turn),tc,float(gd),float(jump))
    return out

def regularize_ridge(F,xy,recovery):
    """MATLAB V6 2-pass Jacobi ridge-constrained corner regularization, raw path preserved."""
    xx=np.asarray(xy,float).copy();ref=xx.copy();changed=np.zeros(len(xx),bool);n=len(xx)
    if n<9:return xx,changed
    for _ in range(2):
        turn=turning(xx[:,0],xx[:,1]);flag=np.zeros(n,bool)
        for i in range(3,n-3):
            if not np.isfinite(turn[i]):continue
            local=turn[max(1,i-2):min(n-1,i+3)];local=local[np.isfinite(local)]
            med=np.median(local) if len(local) else 0
            if turn[i]>=78 or (turn[i]>=30 and turn[i]>=med+6):flag[max(3,i-2):min(n-3,i+3)]=True
        for i in np.flatnonzero(recovery):flag[max(3,i-3):min(n-3,i+4)]=True
        if not flag.any():break
        old=xx.copy()
        for i in np.flatnonzero(flag):
            x,y=old[i]
            t=unit(tangent(*old[i-2],*old[i+2],y))
            if not np.isfinite(t).all():continue
            prev=tangent(x,y,*old[i-1],(y+old[i-1,1])/2)
            nxt=tangent(x,y,*old[i+1],(y+old[i+1,1])/2)
            delta=.62*.5*(prev+nxt)
            dx,dy=km2deg(delta[0],delta[1],y);xt=x+dx;yt=y+dy
            off,xc,yc,z=F.profile(xt,yt,[-t[1],t[0]],18,1.5,('g','gc','a'))
            good=np.isfinite(z['g'])&np.isfinite(z['gc'])&np.isfinite(z['a'])&(z['g']>0)&(z['gc']>0)
            if not good.any():continue
            graw=F.get('g',y,x);gcraw=F.get('gc',y,x)
            if not np.isfinite(graw) or graw<=0:graw=.35*F.g95
            if not np.isfinite(gcraw) or gcraw<=0:gcraw=.35*F.gc95
            aa=F.get('a',old[i-1,1],old[i-1,0]);bb=F.get('a',old[i+1,1],old[i+1,0]);oldturn=turn3(*old[i-1],*old[i],*old[i+1])
            if not np.isfinite(oldturn):oldturn=180
            bestscore=-np.inf;best=None
            for j in np.flatnonzero(good):
                move=geo(ref[i,0],ref[i,1],xc[j],yc[j])
                if move>14 or z['g'][j]<.68*graw or z['gc'][j]<.55*gcraw:continue
                da=max((abs(z['a'][j]-aa) if np.isfinite(aa) else 0),(abs(z['a'][j]-bb) if np.isfinite(bb) else 0))
                if da>.1:continue
                newturn=turn3(*old[i-1],xc[j],yc[j],*old[i+1])
                if not np.isfinite(newturn):continue
                if oldturn-newturn<1.5 and oldturn<78:continue
                if newturn>oldturn+1:continue
                targetdist=geo(xt,yt,xc[j],yc[j])
                sc=min(z['g'][j]/F.g95,1.8)+.55*min(z['gc'][j]/F.gc95,1.8)
                sc-=.42*(targetdist/18)**2+.75*(newturn/90)**2+.35*(da/.1)**2+.2*(move/14)**2
                if sc>bestscore:bestscore=sc;best=(xc[j],yc[j],newturn)
            if best is not None and best[2]<=oldturn:
                xx[i,:2]=best[:2]
                if geo(*old[i],*xx[i])>.25:changed[i]=True
    return xx,changed

def track_adt_v6(lon,lat,adt,config=V6()):
    """Return (axis[N,2], metadata). Axis is surface ADT gradient-front proxy, not 50m velocity core."""
    F=Fields(lon,lat,adt,build_topo=config.use_topology)
    seed=taiwan_seed(F)
    axis=seed.tolist();recs=[False]*len(axis)
    t=unit(tangent(*axis[max(0,len(axis)-4)],*axis[-1],axis[-1][1]))
    if not np.isfinite(t).all():
        gx=F.get('gx',axis[-1][1],axis[-1][0]);gy=F.get('gy',axis[-1][1],axis[-1][0]);t=unit([-gy,gx])
    if not np.isfinite(t).all():raise RuntimeError('ADT V6 seed tangent undefined')
    if t[1]<0:t=-t
    stage=0;fails=0;width=48.;reason='maximum point number reached'
    for _ in range(config.max_points-len(axis)):
        x,y=axis[-1];a0=F.get('a',y,x)
        while stage<3 and gate_pass(x,y,GATES[stage],[.35,.35,.55][stage]):stage+=1
        g=guide(x,y,stage);blend=.18 if stage>=3 else .12
        tpred=unit((1-blend)*t+blend*g)
        dx,dy=km2deg(12*tpred[0],12*tpred[1],y);xp=x+dx;yp=y+dy
        if not F.inside(xp,yp):reason='predicted point left domain';break
        off,xx,yy,z=F.profile(xp,yp,[-tpred[1],tpred[0]],width,3)
        good=np.isfinite(z['g'])&np.isfinite(z['gc'])&np.isfinite(z['a'])&np.isfinite(z['gx'])&np.isfinite(z['gy'])&(z['g']>0)
        cand=ridge_band(z['g'],good,3,.8,.55,15)
        if len(cand)>18:
            merit=z['g'][cand]/F.g95+.45*z['gc'][cand]/F.gc95-.08*np.abs(off[cand])/width
            cand=cand[np.argsort(-merit)[:18]]
        best=None;bs=-np.inf
        for j in cand:
            xc,yc=float(xx[j]),float(yy[j]);gc=float(z['g'][j]);gcc=float(z['gc'][j]);ac=float(z['a'][j])
            tc=unit([-z['gy'][j],z['gx'][j]])
            if not np.isfinite(tc).all():continue
            turn=ang(t,tc)
            if turn>85:continue
            gd=cosangle(tc,g)
            if x<124 and (tc[1]<.05 or gd<-.25):continue
            if 124<=x<130 and gd<-.45:continue
            if x>=130 and (tc[0]<-.45 or gd<-.5):continue
            tok=tokexit=tweff=twret=twturn=twtopo=0.
            if 128.5<=x<=132.5:
                if tc[0]<-.45:continue
                if tc[0]<-.18:tok=((-.18-tc[0])/.27)**2
                if len(axis)>8:
                    old=axis[-8];d=(xc-old[0])*KMLAT*np.cos(np.deg2rad((yc+old[1])/2))
                    if d< -5:tok+=min(2.,(-5-d)/35)
            if stage==0 and 128.7<=x<=131.7:
                if yc>=30.45 and xc<130.25:continue
                if yc>=29.7 and xc<130.05:tokexit+=((130.05-xc)/.55)**2+((yc-29.7)/.9)**2
                if y>=29.7:
                    if tc[0]<-.02:continue
                    if tc[0]<.18:tokexit+=((.18-tc[0])/.20)**2
            if 122<=x<=126.2 and 24.4<=y<=26.9 and len(axis)>7:
                old=axis[-7];ch=geo(old[0],old[1],xc,yc)
                seg=np.asarray(axis[-7:]+[[xc,yc]])
                plen=np.sum(geo(seg[:-1,0],seg[:-1,1],seg[1:,0],seg[1:,1]))
                ef=ch/max(plen,1e-10)
                if ef<.48:continue
                if ef<.72:tweff=((.72-ef)/.24)**2
                dw=(xc-old[0])*KMLAT*np.cos(np.deg2rad((yc+old[1])/2))
                if dw<-12:continue
                if dw<-2:twret=((-2-dw)/10)**2
                if turn>68:continue
                if turn>42:twturn=((turn-42)/26)**2
                if F.topo:
                    closed,known=F.topo.query(xc,yc,ac)
                    twtopo=1.35 if closed and known else (.05 if not known else 0)
            da=abs(ac-a0);relaxed=False;supp=0
            if da>.11:
                if da>.175 or gc/F.g95<.45 or gcc/F.gc95<.35:continue
                supp=support_multi(F,xc,yc,tc)
                if supp<.38:continue
                relaxed=True
            closed,known=F.topo.query(xc,yc,ac) if F.topo and x>=126 else (False,False)
            tp=1.15 if closed and known else (.1 if F.topo and x>=126 and not known else 0)
            loop=0.
            if len(axis)>15:
                old=np.asarray(axis[:-10]);dmin=float(np.min(geo(xc,yc,old[:,0],old[:,1])))
                if dmin<38:loop=1+(38-dmin)/38
            prog=0.
            if x>=128.5 and len(axis)>10:
                old=axis[-9];dnet=(xc-old[0])*KMLAT*np.cos(np.deg2rad((yc+old[1])/2))
                if dnet<-5:prog=min(2,(-5-dnet)/45)
            dx2,dy2=km2deg(12*tc[0],12*tc[1],yc)
            cc=F.get('gc',yc+dy2,xc+dx2)
            look=min(cc/F.gc95,1.5) if np.isfinite(cc) else 0.
            if relaxed:look=max(look,min(supp,1.5))
            sc=min(gc/F.g95,1.5)+.45*min(gcc/F.gc95,1.5)+.25*look
            sc-=.25*(off[j]/width)**2+.65*(turn/60)**2
            sc-=.8*(da/(.11 if relaxed else .05))**2+.35*(max(0,.15-gd)/1.15)**2
            sc-=2.8*loop+1.2*prog+1.35*tok+2.4*tokexit+1.1*tweff+1.15*twret+1.2*twturn+tp+twtopo
            sc+=.08*max(gd,0)
            if sc>bs:bs=sc;best=(xc,yc,gc,ac,float(off[j]),turn,tc,gd,np.nan)
        used=False
        if best is None:
            fails+=1
            if config.enable_recovery and fails>=2:
                best=recover(F,x,y,a0,t,g,axis,stage)
                if best is not None:used=True
            if best is None:
                if fails>=6:reason='six local searches and recovery failed';break
                width=min(width+4,60.);continue
        fails=0;width=48.
        xc,yc,gc,ac,off,turn,tc,gd,jump=best
        axis.append([xc,yc]);recs.append(used)
        t=unit((1-(.50 if used else .30))*t+(.50 if used else .30)*tc)
        if xc>=config.stop_lon and yc>=config.stop_lat and stage>=3:
            reason='eastern downstream target reached';break
    raw=np.asarray(axis,float);rec=np.asarray(recs,bool)
    if config.regularize and len(raw)>=9:final,changed=regularize_ridge(F,raw,rec)
    else:final=raw.copy();changed=np.zeros(len(raw),bool)
    meta={'stop_reason':reason,'success':bool(reason=='eastern downstream target reached'),
          'n_points':int(len(raw)),'last_lon':float(raw[-1,0]),'last_lat':float(raw[-1,1]),
          'n_recovery':int(rec.sum()),'n_regularized':int(changed.sum()),
          'ridge_g95':float(F.g95),'ridge_gc95':float(F.gc95),'method':'ADT_V6_PythonPort'}
    return final,meta,raw,rec,changed


def read_adt_netcdf(path,expected_date=None,lon_min=120.,lon_max=150.,lat_min=20.,lat_max=40.):
    """Lazily subset each daily NetCDF to ADT study window before computing, verifying the actual time."""
    with xr.open_dataset(path,decode_times=True,mask_and_scale=True) as ds:
        names=ds.variables
        lonkey=next((k for k in ('longitude','lon') if k in names),None)
        latkey=next((k for k in ('latitude','lat') if k in names),None)
        adtkey=next((k for k in ('adt','ADT') if k in names),None)
        if not (lonkey and latkey and adtkey):raise KeyError('Expected adt, longitude, latitude; got '+str(list(names)))
        z=ds[adtkey]
        for key,lo,hi in ((lonkey,lon_min,lon_max),(latkey,lat_min,lat_max)):
            coords=ds[key].values
            if coords.ndim!=1:raise ValueError('Only 1D longitude/latitude supported')
            sel=slice(lo,hi) if coords[0]<coords[-1] else slice(hi,lo)
            z=z.sel({key:sel})
        other=[k for k in z.dims if k not in (lonkey,latkey)]
        for k in other:
            if z.sizes[k]==1: z=z.isel({k:0});continue
            if expected_date is not None and k in ds.coords:
                try:
                    dt=pd.to_datetime(ds[k].values)
                    target=pd.Timestamp(expected_date)
                    dif=np.abs(dt-target)
                    ix=int(dif.argmin())
                    if dif[ix]>pd.Timedelta(hours=18):raise ValueError('ADT time mismatch')
                    z=z.isel({k:ix});continue
                except Exception as ex:raise ValueError(f'Cannot confirm time coordinate {k}: {ex}') from ex
            raise ValueError(f'Unexpected non-singleton dimension {k}={z.sizes[k]}')
        if expected_date is not None and 'time' in ds.coords and ds['time'].size==1:
            # Do not silently trust a stale name if time coordinate contradicts it.
            tc=pd.to_datetime(np.asarray(ds['time'].values).reshape(-1)[0]);assert abs(tc-pd.Timestamp(expected_date))<=pd.Timedelta(hours=18),f'time mismatch {tc} vs {expected_date}'
        z=z.transpose(latkey,lonkey)
        lon=np.asarray(z[lonkey].values,float);lat=np.asarray(z[latkey].values,float);a=np.asarray(z.values,float)
        if lon[0]>lon[-1]:lon=lon[::-1];a=a[:,::-1]
        if lat[0]>lat[-1]:lat=lat[::-1];a=a[::-1]
        a[np.abs(a)>100]=np.nan
        if min(len(lon),len(lat))<10 or np.isfinite(a).mean()<.3:raise ValueError('ADT subset too small or mostly invalid')
        return lon,lat,a


In [ ]:
# 5. 仅在 2023 原始失败日上重跑不变V6，保存不完整轨迹供断面保留审计
part_log=TBL/'06_2023部分ADT轨迹状态.csv'
seen=set()
if part_log.exists():
 previous=pd.read_csv(part_log)
 seen=set(previous.loc[previous['status'].isin(['cached','partial_exploratory']),'date'].astype(str))
else:previous=pd.DataFrame()
items=failure.sort_values('date')
if PARTIAL_MAX_DAYS is not None:items=items.iloc[:int(PARTIAL_MAX_DAYS)]
partial_logs=[]
if DO_2023_PARTIAL:
 for idx,row in enumerate(items.itertuples(index=False),1):
  day=str(row.date);stem=day.replace('-','')
  out=PART/(stem+'.npz')
  if day in seen and out.exists():continue
  t0=time.perf_counter()
  try:
   f=Path(row.source_nc)
   if not f.exists():
    raise FileNotFoundError(f'ADT原始文件未挂载: {f}')
   lon,lat,adt=read_adt_netcdf(f,day)
   axis,meta,raw,recovery,reg=track_adt_v6(lon,lat,adt,V6())
   if meta['success']:
    status='unexpected_full_success' # 绝不自动转为主分析！需要调查算法不确定性
   elif len(axis)<20 or not np.isfinite(axis).all():
    status='invalid_partial'
   else:
    status='partial_exploratory'
    np.savez_compressed(out,axis_lon=axis[:,0].astype('float32'),axis_lat=axis[:,1].astype('float32'),
      date=np.array(day),source=np.array('R72_ADT_V6_UNCHANGED_PARTIAL_EXPLORATORY'))
   line={'date':day,'status':status,'n_points':len(axis),'last_lon':float(axis[-1,0]) if len(axis) else np.nan,
    'last_lat':float(axis[-1,1]) if len(axis) else np.nan,'stop_reason':meta.get('stop_reason',''),
    'source_nc':str(f),'elapsed_s':round(time.perf_counter()-t0,2)}
  except Exception as e:line={'date':day,'status':'error','error':repr(e),'elapsed_s':round(time.perf_counter()-t0,2)}
  partial_logs.append(line)
  if len(partial_logs)>=20 or idx==len(items):
   previous=pd.concat([previous,pd.DataFrame(partial_logs)],ignore_index=True)
   previous=previous.drop_duplicates('date',keep='last').sort_values('date')
   save_csv(previous,'06_2023部分ADT轨迹状态.csv');partial_logs.clear()
  if idx%30==0:print('[Partial 2023]',idx,'/',len(items),flush=True)
print('Partial track files:',len(list(PART.glob('*.npz'))),'original failed days:',len(failure))


In [ ]:
# 6. 六断面：从ADT单独确定交点，所有交点与终止位置关系须事先验证
SECTIONS={
 'A':((121.50,23.50),(125.00,22.75)),
 'B':((121.50,26.25),(124.25,24.25)),
 'PN':((126.00,29.00),(128.15,27.30)),
 'C':((130.25,31.25),(132.50,27.00)),
 'D':((137.00,34.75),(138.00,27.75)),
 'E':((141.75,40.00),(145.00,31.50))}

def km_xy(a,lat0=31.0):
 x=np.asarray(a,float)
 return np.stack([x[...,0]*111.195*np.cos(np.deg2rad(lat0)),x[...,1]*111.195],axis=-1)
def cross2(a,b):return float(a[0]*b[1]-a[1]*b[0])
def section_all(axis,sec):
 p1,p2=sec
 xy=km_xy(axis);r=km_xy(np.array([p1,p2]));R=r[1]-r[0]
 ans=[]
 for i in range(len(axis)-1):
  p=xy[i];v=xy[i+1]-p;D=cross2(v,R)
  if abs(D)<1e-10:continue
  o=r[0]-p;t=cross2(o,R)/D;u=cross2(o,v)/D
  if -1e-9<=t<=1+1e-9 and -1e-9<=u<=1+1e-9:
   q=axis[i]+np.clip(t,0,1)*(axis[i+1]-axis[i]);distance=float(gc_km(np.linalg.norm(xyz(np.array([q]))[0]-xyz(np.array([p1]))[0])))
   if not any(abs(q0['distance_km']-distance)<.05 for q0 in ans):
    ans.append({'distance_km':distance,'lon':float(q[0]),'lat':float(q[1]),'index':i+float(t)})
 return ans

# 三个先验限制：不完整的ADT路径必须 >=20点、交点后 >=3点、轨迹没有大于250km单段断裂。
sec_rows=[]
for axisfile in sorted(PART.glob('*.npz')):
 day=pd.Timestamp(axisfile.stem).strftime('%Y-%m-%d')
 try:
  a=load_axis(axisfile,'ADT')
  if len(a)<20 or (gc_km(np.linalg.norm(np.diff(xyz(a),axis=0),axis=1))>250).any():continue
  for sec,loc in SECTIONS.items():
   matches=section_all(a,loc)
   valid=[r for r in matches if r['index']<=len(a)-1-MIN_SECTION_REMAINING_POINTS]
   x=valid[0] if valid else None
   sec_rows.append({'date':day,'section':sec,'source':'ADT_partial_exploratory',
     'n_intersections':len(matches),'n_safe_intersections':len(valid),'safe':x is not None,
     'distance_km':x['distance_km'] if x else np.nan,
     'selected_lon':x['lon'] if x else np.nan,'selected_lat':x['lat'] if x else np.nan,
     'last_lon':float(a[-1,0]),'last_lat':float(a[-1,1]),'n_points':len(a)})
 except Exception as e:
  sec_rows.append({'date':day,'section':'ERROR','source':'ADT_partial_exploratory','error':str(e),'safe':False})
partial_secs=pd.DataFrame(sec_rows)
if not partial_secs.empty:
 save_csv(partial_secs,'07_2023部分ADT轨迹六断面候选.csv')
 print('ADT-only safe partial coverage by section')
 display(partial_secs.groupby('section').agg(partial_days=('date','nunique'),safe_intersection_days=('safe','sum')))
else:print('No partial output; check DO_2023_PARTIAL and source data path')


In [ ]:
# 7. 与2023原全程成功日期组合比较“每断面可用性”；候选并不进入论文主表
coverage=[]
for sec in SECTIONS:
 original_days=set(section.loc[(section.section==sec)&(section.date.astype(str).str.startswith('2023'))&section.ADT.notna(),'date'].astype(str))
 partial_days=set(partial_secs.loc[(partial_secs.section==sec)&(partial_secs.safe),'date'].astype(str)) if not partial_secs.empty else set()
 coverage.append({'section':sec,'2023_total_days':365,'full_valid_days':len(original_days),
    'partial_adt_only_candidate_days':len(partial_days),
    'combined_exploratory_potential_days':len(original_days|partial_days),
    'is_official_validation':False})
cov=pd.DataFrame(coverage)
save_csv(cov,'08_2023六断面潜在覆盖改善_仅探索.csv')
display(cov)
# 注意：这里仅报告ADT数据本身的覆盖，尚未检验预测模型，也不因GLORYS一致而筛选候选。


In [ ]:
# 8. 年份统计/图表/完整归档
plt.rcParams.update({'font.size':10,'axes.grid':True,'grid.alpha':.2})
if 'v' in globals() and len(v):
 fig,ax=plt.subplots(figsize=(10.5,4),dpi=180)
 g=v.groupby('year')[['original_rmse_km','common_rmse_km']].mean()
 ax.plot(g.index,g.original_rmse_km,label='Uncropped',lw=1.5)
 ax.plot(g.index,g.common_rmse_km,label='Common 144.3 E',lw=1.7)
 ax.set(xlabel='Year',ylabel='Mean daily axis RMSE (km)',title='ADT vs GLORYS: common spatial support')
 ax.legend();fig.tight_layout();fig.savefig(FIG/'Fig01_AllDays_CommonDomain_byYear.png',dpi=220);fig.savefig(FIG/'Fig01_AllDays_CommonDomain_byYear.pdf');plt.close(fig)
fig,ax=plt.subplots(figsize=(10.5,3.5),dpi=180)
g=qc.copy();g['year']=g.date.astype(str).str[:4].astype(int)
s=g.groupby('year').status.apply(lambda z:float(np.mean(z=='success')))
ax.bar(s.index,s.values,width=.85)
ax.set(xlabel='Year',ylabel='ADT full-track success fraction',ylim=(0,1.05),title='ADT V6 tracking coverage (1993–2023)')
fig.tight_layout();fig.savefig(FIG/'Fig02_ADT_Coverage_byYear.png',dpi=220);fig.savefig(FIG/'Fig02_ADT_Coverage_byYear.pdf');plt.close(fig)
fig,ax=plt.subplots(figsize=(9,3.8),dpi=180)
ax.bar(cov.section,cov.full_valid_days,label='Complete ADT (primary)')
ax.bar(cov.section,cov.partial_adt_only_candidate_days,bottom=cov.full_valid_days,
 label='Failed-track prefix (exploratory)',alpha=.6)
ax.set(xlabel='Section',ylabel='Available 2023 days',title='2023 section coverage: incomplete tracks never counted as complete')
ax.legend(fontsize=8);fig.tight_layout();fig.savefig(FIG/'Fig03_2023_PartialSectionCoverage.png',dpi=220);fig.savefig(FIG/'Fig03_2023_PartialSectionCoverage.pdf');plt.close(fig)

status={
 'r70_main_checksum':sha256(R70_RESULTS),'r70_qc_checksum':sha256(R70_QC),
 'r70_adt_total_days':len(qc),'r70_full_adt_days':int((qc.status=='success').sum()),
 'common_domain_all_requested':DO_ALL_COMMON,
 'common_domain_days_completed':int(len(v)) if 'v' in globals() else 0,
 'common_domain_total_target':10279,
 '2023_failed_days':len(failure),
 '2023_original_full_valid_days':91,
 '2023_partial_days_reprocessed':int(len(partial_secs.date.unique())) if not partial_secs.empty else 0,
 '2023_partial_track_count':len(list(PART.glob('*.npz'))),
 '2023_partial_max_days_setting':PARTIAL_MAX_DAYS,
 'partial_is_not_independent_full_validation':True,
 'forecast_evaluation_performed':False,
 'scientific_limitations':[
 'R7.2 did not use GLORYS for selecting/tracking ADT reference paths',
 'Partial ADT tracks from failed days are exploratory and may contain errors',
 'Forecast-to-ADT evaluation requires real checkpoint-based forecast paths',
 'GLORYS 50m subsurface core and satellite surface front differ physically',
 'Satellite altimetry may be assimilated in GLORYS; partial non-independence',
 'Python V6 port is not validated numerically against MATLAB V6',
 'Strong 2023 missingness biases available-case trend if ignored']}
(REP/'R72_实验状态及科学边界.json').write_text(json.dumps(status,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
(REP/'R72_结果释义.txt').write_text("R7.2: primary reference = original completed ADT axes only.\n" +
 "Common-domain scores use original 10,279 accepted tracks; 2023 incomplete ADT paths are NOT merged into primary path RMSE.\n" +
 "2023 ADT-only incomplete-track section candidates are exploratory; require independent satellite physical QC before publication.\n" +
 "No prediction checkpoint was loaded. Forecast vs ADT has NOT been evaluated.\n", encoding='utf-8')

archive=ROOT/'R72_上传给ChatGPT_全量共域与2023断面审计.zip'
# 缓存的2023大量单日npz保留在用户本地，不打包给聊天，ZIP仅包含CSV报告+论文图。
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zz:
 for folder in (TBL,FIG,REP):
  for p in sorted(folder.rglob('*')):
   if p.is_file():zz.write(p,p.relative_to(OUT))
with zipfile.ZipFile(archive) as z:assert z.testzip() is None
Path(str(archive)+'.sha256').write_text(sha256(archive)+'  '+archive.name+'\n',encoding='utf-8')
print('R7.2 output ZIP:',archive,'SHA256:',sha256(archive))
display(FileLink(str(archive)))
